# 03 · Baselines: how far do simple rules get?

Before any machine learning, we measure **rules that need no training**. If a model can't beat them, it isn't worth deploying.

### Metrics (how the benchmark scores RCA)
For each case, sort services by score. Find the position (rank) of the true root cause.
- **AC@1** = share of cases where the true cause is ranked **first** (top-1 accuracy)
- **AC@3 / AC@5** = share where it is in the top 3 / top 5
- **Avg@5** = average of AC@1…AC@5 (the RCAEval summary number)
- **MRR** = average of 1 / rank

Ties are counted *pessimistically* (if the culprit shares the top score with 4 others, it counts as rank 5), so a rule cannot look good by luck.

### The rules
| Rule | Score of a service |
|---|---|
| random | random number (expected value shown) |
| **max metric z (BARO-style)** | its largest robust z-score over all its metrics |
| latency only | largest z of latency-50/90 |
| resource only | largest z of cpu / mem / diskio / socket |
| log errors | increase in error-like log lines |
| trace p95 | increase in p95 span latency (OB, TT only) |
| rank average | average of its within-case ranks over all features |

These rules use **all 270 cases** (no training, so no split needed).

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*(mismatched devices|pandas version 4).*")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "reports" / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

from rcalens.evaluate import evaluate, true_ranks
df = pd.read_parquet(ROOT / "data" / "processed" / "service_features.parquet")
has_traces = bool(df.columns.str.startswith("t_").any())

# one score column per rule: higher = more likely the root cause
scores = pd.DataFrame({
    "max metric z (BARO-style)": df["m_all_maxz"],
    "latency only": df[["m_latency50_maxz", "m_latency90_maxz"]].max(axis=1),
    "resource only (cpu/mem/disk/socket)": df[["m_cpu_maxz", "m_mem_maxz", "m_diskio_maxz", "m_socket_maxz"]].max(axis=1),
    "log errors": df["l_err_ratio"],
    "rank average (all features)": df.loc[:, df.columns.str.endswith("_rank")].mean(axis=1),
})
if has_traces:
    scores["trace p95"] = df["t_p95_ratio"]
scores = scores.fillna(-1e9)                              # no evidence -> bottom of the ranking

def scored(rule):
    return df[["case", "system", "service", "is_rc"]].assign(score=scores[rule])

# random guessing: expected metrics from the number of candidates n of each case
n = df.groupby("case").size()
top_k = pd.DataFrame({k: np.minimum(k, n) / n for k in range(1, 6)})        # chance that a random top k contains the culprit
random_guess = pd.Series({"AC@1": top_k[1].mean(), "AC@3": top_k[3].mean(), "AC@5": top_k[5].mean(),
                          "Avg@5": top_k.mean(axis=1).mean(),
                          "MRR": n.map(lambda k: (1 / np.arange(1, k + 1)).mean()).mean()})

overall, per_system = {"random (expected)": random_guess}, {}
for rule in scores.columns:
    ev = evaluate(scored(rule))                         # one row for all cases + one per system
    overall[rule] = ev.loc["all"]
    per_system[rule] = ev["AC@1"]
base = pd.DataFrame(overall).T
base.round(3)

**Reading it:** compare each rule with *random*. The strongest single rule is the one to beat in notebook 04.

### Per system (AC@1, top-1 accuracy)

In [ ]:
pd.DataFrame(per_system).T.round(3)

### Per fault type (AC@1 of the best rule)
Which faults are easy and which are hard for the rule?

In [ ]:
best_rule = base.drop(index="random (expected)")["AC@1"].idxmax()
print("best rule:", best_rule)
top1 = (true_ranks(scored(best_rule)) <= 1).rename("AC@1").to_frame()       # one row per case: culprit ranked first?
top1 = top1.join(df.drop_duplicates("case").set_index("case")["fault"])
top1.groupby("fault")["AC@1"].mean().round(3).to_frame().T

In [ ]:
json.dump({"has_traces": has_traces, "best_rule": best_rule, "overall": base.round(4).to_dict(orient="index"),
           "ac1_by_system": pd.DataFrame(per_system).round(4).to_dict()},
          open(RESULTS / "baselines.json", "w"), indent=1)
scored(best_rule).to_parquet(ROOT / "data" / "processed" / "baseline_scores.parquet", index=False)
print("saved reports/results/baselines.json")

### Takeaways
- A single robust statistic (the BARO idea) is already far above random: injected faults usually leave a clear mark on the culprit's own metrics.
- It struggles where the symptom *travels*: delay and loss faults make callers slow too, so the slowest service is often not the cause.
- That's the opening for a learned model that combines many weak signals (resource vs latency, logs, traces, ranks).

➡️ Next: **04_ml_ranker**: learning to rank with XGBoost.